In [ ]:
# Verificar o status da GPU (se aplicável)
!nvidia-smi

In [ ]:
# Garantir que estamos no diretório base e que não existe um yolov5 anterior
%cd /content
!rm -rf yolov5  # Remove qualquer clone anterior

# Clonar o repositório YOLOv5
!git clone https://github.com/ultralytics/yolov5
%cd yolov5

# Instalar dependências do YOLOv5
%pip install -qr requirements.txt
%pip install -q roboflow

In [ ]:
from google.colab import drive
import os

# Montar o Google Drive
drive.mount('/content/drive')

In [ ]:
# Desmontar o Google Drive
# drive.flush_and_unmount()

In [ ]:
import shutil
import yaml

# 1. Caminhos
caminho_zip_drive = "/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Yolo_dataset.zip"
pasta_local_vm = "/content/Yolo_dataset"

# 2. Criar a pasta "caixinha" para não espalhar os arquivos
os.makedirs(pasta_local_vm, exist_ok=True)

# 3. Automação MLOps (Verifica se o YAML já está lá dentro)
if not os.path.exists(f"{pasta_local_vm}/data.yaml"):
    print("🚚 1. Copiando o ZIP do Drive para o SSD da VM...")
    shutil.copy2(caminho_zip_drive, "/content/Yolo_dataset.zip")
    
    print("📦 2. Descompactando na velocidade da luz...")
    # AGORA SIM! Descompactando para DENTRO da pasta_local_vm
    shutil.unpack_archive("/content/Yolo_dataset.zip", pasta_local_vm)
    
    print("🔧 3. Ajustando o data.yaml para o caminho do SSD local...")
    yaml_path = f"{pasta_local_vm}/data.yaml"
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    data['path'] = pasta_local_vm
    
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, sort_keys=False)
        
    print("🚀 SUCESSO! Ambiente efêmero montado e pronto para treinar.")
else:
    print("✅ O dataset já está no SSD desta sessão e pronto para uso!")

# 4. Aponta a variável global do YOLO para o SSD local
os.environ["DATASET_DIRECTORY"] = pasta_local_vm

In [ ]:
import torch

print(f"Setup completo. Usando torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

# Destrói qualquer arquivo de cache corrompido do YOLO nas pastas
!rm -f "$DATASET_DIRECTORY/labels/train/*.cache"
!rm -f "$DATASET_DIRECTORY/labels/val/*.cache"
!rm -f "$DATASET_DIRECTORY/images/train/*.cache"
!rm -f "$DATASET_DIRECTORY/images/val/*.cache"

# Desativa o Weights & Biases para evitar o prompt interativo no notebook
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
print("W&B desativado para este notebook.")

# Executa o treinamento com a variável de ambiente
# !python train.py --batch 16 --epochs 1 --data "$DATASET_DIRECTORY/data.yaml" --weights yolov5s.pt --cache
!python train.py --batch 64 --epochs 5 --data "$DATASET_DIRECTORY/data.yaml" --weights yolov5s.pt --workers 4 --cache disk


In [ ]:
import shutil

# Caminho de origem (onde o YOLO salvou)
origem = "/content/yolov5/runs/train/exp3"

# Caminho de destino (O seu Google Drive)
# Mudei o final para criar uma pasta "Resultados_YOLO"
destino = "/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Resultados_YOLOv5"

# Copia a pasta inteira
shutil.copytree(origem, destino)

print("✅ Todos os gráficos e o modelo 'best.pt' foram salvos no seu Drive com sucesso!")

In [ ]:
# Start tensorboard
# Launch after you have started training
# logs save in the folder "runs"
# %load_ext tensorboard
# %tensorboard --logdir runs